# Evo 2 DNA Sequence Generation and Scoring

![Evo 2 DNA Sequence Generation and Scoring](https://proto-bio.github.io/proto-assets/images/tool/evo2/hero.png)

This notebook demonstrates examples of how to use the Evo 2 tools. In the first section, we use `run_evo2_sample` to extend a short DNA prompt into a longer sequence using autoregressive sampling with KV caching for efficient generation. In the second section, we use `run_evo2_score` to compute log-likelihood and perplexity metrics that quantify how natural a given sequence appears to the model. A final section uses `run_evo2_icl` to condition new DNA designs on examples in a FASTA file.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("evo2")
display_overview("evo2")
display_docs_section("evo2", "Background")

## Available tools

In [ ]:
display_available_tools("evo2")

### `run_evo2_sample`

This tool utilizes Evo 2 generates DNA sequences autoregressively from a starting prompt, extending a sequence nucleotide-by-nucleotide according to the model's learned distribution over genomic DNA. The `temperature` and `top_k` parameters control sampling diversity: lower values produce conservative, high-confidence extensions while higher values allow more exploratory generation.

In [ ]:
from proto_tools import (
    Evo2SampleInput,
    Evo2SampleConfig,
    run_evo2_sample,
)

In [ ]:
# Display docs
display_api_reference("evo2", "input", "run_evo2_sample")

# Seed generation with a real E. coli lacZ coding-sequence fragment.
inputs = Evo2SampleInput(prompts=["ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGG"])

In [ ]:
# Display config docs
display_api_reference("evo2", "config", "run_evo2_sample")

# Configure sampling parameters | see docs above for all fields
config = Evo2SampleConfig(
    max_new_tokens=200,
    temperature=0.8,
    top_k=4,
    device="cuda",
)

In [ ]:
# Run the sampling function
sample_result = run_evo2_sample(inputs, config)

In [ ]:
# Display output docs
display_api_reference("evo2", "output", "run_evo2_sample")

# Show the prompt and the generated continuation
prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} nucleotides")

### `run_evo2_score`

Evo 2 scores sequences by computing the autoregressive log-likelihood at each position, measuring how predictable each nucleotide is given its preceding context. The resulting perplexity provides a single-number summary of sequence naturalness: lower perplexity means the sequence more closely matches the statistical patterns in the training data. This is useful for ranking generated candidates, comparing wild-type and engineered variants, or filtering out sequences that deviate substantially from natural genomic patterns.

In [ ]:
from proto_tools import (
    Evo2ScoringInput,
    Evo2ScoringConfig,
    run_evo2_score,
)

In [ ]:
# Display docs
display_api_reference("evo2", "input", "run_evo2_score")

# Contrast a natural coding sequence with a shuffled control of identical composition;
# Evo2 should assign the natural sequence the lower (better) perplexity.
score_inputs = Evo2ScoringInput(sequences=["ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGG", "GACGTTCATGCACTAGGTTGATCTGTCCGATAAACTGGCATGCCTTAGCTG"])

In [ ]:
# Display config docs
display_api_reference("evo2", "config", "run_evo2_score")

# Configure scoring | see docs above for all fields
score_config = Evo2ScoringConfig(
    batch_size=2,
    return_logits=False,
    device="cuda",
)

In [ ]:
# Run the scoring function
score_result = run_evo2_score(score_inputs, score_config)

In [ ]:
# Display output docs
display_api_reference("evo2", "output", "run_evo2_score")

# Show scoring results for each input sequence
for i, seq in enumerate(score_inputs.sequences):
    score = score_result.scores[i]
    print(f"Sequence {i + 1}: {seq}")
    print(f"    Log-likelihood:     {score.log_likelihood:.3f}")
    print(f"    Avg log-likelihood: {score.avg_log_likelihood:.3f}")
    print(f"    Perplexity:         {score.perplexity:.3f}")

## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for use in downstream analysis pipelines. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sample_result.export(name="evo2_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'evo2_sequences.fasta'}")

score_result.export(name="evo2_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'evo2_scores.csv'}")

## In-context DNA design (`run_evo2_icl`)

In-context learning conditions pretrained Evo 2 7B on example sequences without updating its weights. The FASTA is an example pool: `n_shots` records are drawn without replacement for each of `n_prompts` prompts. The default separator is a single `N`, including a leading and trailing `N`. Generated designs exclude the examples and separators.

The short records below illustrate the format. A real [HepG2 pool of 463 200-bp sequences](https://github.com/evo-design/ICL_regulatory/blob/main/inference/data/invitro_cell_activity_seqprop/HepG2.fasta) can replace them. The ICL paper URL remains a placeholder.


In [ ]:
from io import StringIO

from Bio import SeqIO
from proto_tools import Evo2ICLInput, Evo2ICLConfig, run_evo2_icl

# Replace this illustrative FASTA with examples sharing the desired sequence family.
example_fasta = ">illustrative_1\nACGTACGTACGT\n>illustrative_2\nTGCATGCATGCA\n"
records = list(SeqIO.parse(StringIO(example_fasta), "fasta"))
# For a local file instead: records = list(SeqIO.parse("prompt.fasta", "fasta"))
icl_inputs = Evo2ICLInput(examples=[str(record.seq) for record in records])
preview_config = Evo2ICLConfig(n_shots=2, n_prompts=2, example_seed=42)
prompts, selected = preview_config.compose_prompts(icl_inputs)
print(prompts, selected)  # CPU-only prompt preview; example positions are 1-indexed.


The defaults request 20 designs of 200 bp with temperature `1.0`, top K `4`, and top P `0.9`. Temperature controls randomness; top K limits the likely tokens considered, and top P limits cumulative probability mass. These controls do not score biological activity. The illustrative 12-bp examples below request matching 12-bp designs; the linked 200-bp HepG2 examples can condition the default 200-bp designs.

Local and user-owned Modal runs have no design-count quota. The web form limits Proto-managed runs to 100 designs per prompt and 1,000 designs across all prompts. There is no application cap on prompt size or aggregate generated bases. Each design runs to its required positive length without early stopping; separators do not stop generation. Evo 2 Flash continuously batches designs and reuses shared prompts; the legacy batch-size and prefill settings are accepted but ignored. The local GPU call below shares the existing Evo 2 worker; set `device="proto"` for Proto's hosted Modal service.


In [ ]:
display_api_reference("evo2", "config", "run_evo2_icl")

icl_config = Evo2ICLConfig(
    n_shots=2,
    n_prompts=2,
    num_sequences=5,  # 10 designs total
    example_seed=42,
    fixed_examples=False,
    spacer="N",
    spacer_both_sides=False,
    sequence_length=12,
    temperature=1.0,
    top_k=4,
    top_p=0.9,
    batch_size=1,
    device="cuda",
    seed=42,
)
icl_result = run_evo2_icl(icl_inputs, icl_config)
print(f"Generated {len(icl_result.sequences)} designs")
for warning in icl_result.warnings:
    print(warning)


### Export ICL designs

FASTA export contains ACGTN designs at the requested length; sampled separators are retained. Wrong-length or invalid generations are omitted with a yield warning. Designs are candidates for downstream evaluation; this operation does not establish regulatory activity or specificity.


In [ ]:
icl_result.export("evo2_icl_designs", export_path=output_dir, file_format="fasta")
print(f"ICL designs exported to {output_dir / 'evo2_icl_designs.fasta'}")
